In [ ]:
# pylint: skip-file

In [ ]:
import torch
import torch.nn as nn
import math

## Multi-head self-attention


In [ ]:
class MHSA(nn.Module):
    def __init__(self, n_chan, n_head):

        self.n_chan = n_chan
        self.n_head = n_head
        self.h_dim = n_chan // n_head

        # [h_dim][h_dim] ... [h_dim] = [h*h_dim] = [n_chan]

        self.Wq = nn.Linear(n_chan, n_chan, bias=False)
        self.Wk = nn.Linear(n_chan, n_chan, bias=False)
        self.Wv = nn.Linear(n_chan, n_chan, bias=False)

        self.Wo = nn.Linear(n_chan, n_chan, bias=False)

    def forward(self, x):
        # x: [n_batch, n_sequenze, n_chan]
        n_batch, n_sequenze, n_chan = x.size()

        Q = self.Wq(
            x
        )  # [n_batch, n_sequenze, n_chan] [n_chan, n_chan] = [n_batch, n_sequenze, n_chan]
        K = self.Wk(x)
        V = self.Wv(x)

        # [n_batch, n_sequenze, n_chan] -> [n_batch, n_sequenze, n_head, h_dim]
        # donde n_chan = n_head * h_dim

        Q = Q.view(n_batch, n_sequenze, self.n_head, self.h_dim).transpose(-2, -3)
        K = K.view(n_batch, n_sequenze, self.n_head, self.h_dim).transpose(1, 2)
        V = V.view(n_batch, n_sequenze, self.n_head, self.h_dim).transpose(1, 2)
        # [n_batch, n_head, n_sequenze, h_dim]

        scores = torch.matmul(Q, K.transpose(-2, -1))
        attn = torch.softmax(scores / math.sqrt(self.h_dim), dim=-1)

        # [n_batch,n_heads,n_sequenze,h_dim]
        out = torch.matmul(attn, V)

        # [n_batch,n_sequenze,n_heads,h_dim]
        out = out.transpose(-2, -3)

        # [n_batch,n_sequenze,n_heads*h_dim] = [n_batch,n_sequenze,n_chan]
        out = out.view(n_batch, n_sequenze, self.n_chan)

        return self.Wo(out)

## Bloque de Pytorch


In [ ]:
n_dim = 256
n_heads = 8

mha = nn.MultiheadAttention(n_dim, n_heads, dropout=0.1, batch_first=True)

# n_dim: Dimensión de los embeddings de entrada (número de features por token).
# n_heads: Número de heads

# bias: Si las proyecciones lineales (Q, K, V, out) tienen bias
#       Se suele fijar en True.
#       Default: True

# dropout: Probabilidad de dropout sobre los pesos de atención.
#          Se suele fijar en 0.1
#          Default: 0.0

# kdim, vdim: Dimensiones de key y value (si son distintas que n_dim)
#             Util para cross-attention
#             Default: None

# batch_first: Si es  True, entrada: [n_batch, n_seq, n_dim]    (lo más usado)
#              Si es False, entrada: [n_seq, n_batch, n_dim]
#              Default: False

In [ ]:
output, attnMap = mha(
    query, key, value, key_padding_mask=None, need_weights=True, attn_mask=None
)

# query, key, value: Las fuentes para calcular las señales query, key, value

# need_weights: Si está activo se devuelve también los pesos de atención.
#               Se suele dejar False en entrenamiento (ahorra memoria/tiempo);
#               y True si quieres visualizar o depurar.
#               Default: True

# attn_mask: Aqui insertamos la mascara causal (Mask Multi-head Self-Attention, Decoder)

## Bloques de Encoder y Decoder


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, n_heads, n_dim):
        super().__init__()

        self.MHSA = nn.MultiheadAttention(n_dim, n_heads)
        self.norm1 = nn.LayerNorm(n_dim)
        self.norm2 = nn.LayerNorm(n_dim)

        self.feedforward = nn.Sequential(
            nn.Linear(n_dim, 4 * n_dim), nn.GeLU(), nn.Linear(4 * n_dim, n_dim)
        )

        self.dropout1 = nn.Dropout(n_dim)
        self.dropout2 = nn.Dropout(n_dim)

    def forward(self, x):
        y, _ = self.MHSA(x, x, x)
        y = self.dropout1(y)
        y = self.norm1(x + y)

        z = self.feedforward(y)
        z = self.dropout2(z)
        return self.norm2(y + z)

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, n_heads, n_dim):
        super().__init__()

        self.MHSA = nn.MultiheadAttention(n_dim, n_heads)
        self.MHCA = nn.MultiheadAttention(n_dim, n_heads)
        self.norm1 = nn.LayerNorm(n_dim)
        self.norm2 = nn.LayerNorm(n_dim)

        self.feedforward = nn.Sequential(
            nn.Linear(n_dim, 4 * n_dim), nn.GeLU(), nn.Linear(4 * n_dim, n_dim)
        )

        self.dropout1 = nn.Dropout(n_dim)
        self.dropout2 = nn.Dropout(n_dim)

    # x: viene del encoder
    # y: son las salidas que estamos generando
    def forward(self, x, y):
        # Self-attention
        k, _ = self.MHSA(y, y, y)
        k = self.dropout1(k)
        y = self.norm1(k + y)

        # Cross-attention
        k, _ = self.MHCA(y, x, x)
        k = self.dropout1(k)
        y = self.norm1(k + y)

        # Feed forward
        z = self.feedforward(y)
        z = self.dropout2(z)
        z = self.norm2(y + z)